# 02 · Construcción del Dataset de Entrenamiento — Savia Salud EPS

**Objetivo:** Extraer un dataset etiquetado con las 3 clases del TARGET desde la BD `system_savia`  
y guardarlo como Parquet para las fases de EDA y Feature Engineering.

## Estrategia de extracción
Para evitar timeouts en JOINs sobre 260M filas (`cm_detalles`), se extrae en **3 pasos independientes**:
1. **IDs con TARGET** — query ligero que clasifica cada factura en {0, 1, 2}
2. **Features de cm_facturas** — columnas disponibles en radicación (sin data leakage)
3. **Features agregadas de cm_detalles** — conteos y sumas por factura_id

## Columnas EXCLUIDAS por data leakage
Las siguientes columnas reflejan el resultado del proceso de auditoría (no disponibles al momento de la radicación):  
`valor_inicial_glosa`, `valor_pagado_factura`, `valor_pendiente_actual`, `estado_factura`,  
`ejecucion_glosa_aut_ia`, `estado_glosa_aut`, `valor_no_acuerdo`, `usuario_audita`

**Salida:** `data/processed/20260304_dataset_entrenamiento.parquet`

In [ ]:
# Celda 1 — Conexión y configuración
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from pathlib import Path
import time

# Credenciales (igual que extraccion_xml_json.ipynb)
user     = "jcardonr"
password = "6nxx2yVWaKXyiF1cZbfS*"
host     = "10.250.5.36"
db_name  = "system_savia"
port     = "3306"

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{db_name}",
    connect_args={"connect_timeout": 30},
    pool_recycle=3600,
)

# Carpetas de salida
PROCESSED_DIR = Path(r"D:\Users\jcardonr\Documents\eafit\Codigo\data\processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = PROCESSED_DIR / "20260304_dataset_entrenamiento.parquet"

# Ventana temporal: últimos 24 meses
MESES_HISTORICO = 24

# Verificar conexión
try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print(f"✅ Conexión exitosa → {host}/{db_name}")
    print(f"📁 Salida: {OUTPUT_FILE}")
except Exception as e:
    print(f"❌ Error de conexión: {e}")

In [ ]:
# Celda 2 — Paso 1: Extraer IDs con TARGET
# Estrategia: 3 queries independientes por clase para evitar timeout.
# La clasificación de prioridad es: Devuelta > Glosada > Auditada.

print("=" * 60)
print("PASO 1: Extrayendo IDs con TARGET")
print("=" * 60)

# --- Clase 2: DEVUELTA ---
# cm_auditoria_devoluciones tiene FK directa a cm_facturas_id
q_devueltas = """
SELECT DISTINCT f.id AS factura_id, 2 AS target
FROM cm_facturas f
INNER JOIN cm_auditoria_devoluciones cad ON cad.cm_facturas_id = f.id
WHERE f.fecha_radicacion >= DATE_SUB(NOW(), INTERVAL %(meses)s MONTH)
"""
print("  Clase 2 (Devuelta): consultando...")
t0 = time.time()
df_devueltas = pd.read_sql(q_devueltas, con=engine, params={"meses": MESES_HISTORICO})
print(f"  → {len(df_devueltas):,} facturas Devueltas ({time.time()-t0:.1f}s)")

# --- Clase 1: GLOSADA ---
# Una factura es Glosada si tiene al menos 1 motivo de glosa en cm_auditoria_motivos_glosas
# y NO está en Devueltas (prioridad de Devuelta > Glosada)
ids_devueltas = tuple(df_devueltas["factura_id"].tolist()) if len(df_devueltas) > 0 else (-1,)

q_glosadas = """
SELECT DISTINCT f.id AS factura_id, 1 AS target
FROM cm_facturas f
INNER JOIN cm_detalles cd ON cd.cm_facturas_id = f.id
INNER JOIN cm_auditoria_motivos_glosas camg ON camg.cm_detalles_id = cd.id
WHERE f.fecha_radicacion >= DATE_SUB(NOW(), INTERVAL %(meses)s MONTH)
  AND f.id NOT IN (SELECT cm_facturas_id FROM cm_auditoria_devoluciones
                    WHERE cm_facturas_id IS NOT NULL)
LIMIT 400000
"""
print("  Clase 1 (Glosada): consultando (puede tardar 2-5 min)...")
t0 = time.time()
df_glosadas = pd.read_sql(q_glosadas, con=engine, params={"meses": MESES_HISTORICO})
print(f"  → {len(df_glosadas):,} facturas Glosadas ({time.time()-t0:.1f}s)")

# --- Clase 0: AUDITADA ---
# Facturas que NO están en Glosadas ni Devueltas
# Usamos una muestra proporcional para no sobrecargar la consulta
# Se toma 1 de cada 15 facturas (≈ 6.7%) → ~250K registros
q_auditadas = """
SELECT f.id AS factura_id, 0 AS target
FROM cm_facturas f
WHERE f.fecha_radicacion >= DATE_SUB(NOW(), INTERVAL %(meses)s MONTH)
  AND NOT EXISTS (SELECT 1 FROM cm_auditoria_devoluciones cad
                   WHERE cad.cm_facturas_id = f.id)
  AND NOT EXISTS (SELECT 1 FROM cm_detalles cd
                   INNER JOIN cm_auditoria_motivos_glosas camg
                     ON camg.cm_detalles_id = cd.id
                   WHERE cd.cm_facturas_id = f.id)
  AND MOD(f.id, 20) = 0
LIMIT 400000
"""
print("  Clase 0 (Auditada): consultando (puede tardar 3-8 min)...")
t0 = time.time()
df_auditadas = pd.read_sql(q_auditadas, con=engine, params={"meses": MESES_HISTORICO})
print(f"  → {len(df_auditadas):,} facturas Auditadas ({time.time()-t0:.1f}s)")

# Combinar IDs con labels
df_ids = pd.concat([df_devueltas, df_glosadas, df_auditadas], ignore_index=True)
df_ids = df_ids.drop_duplicates(subset=["factura_id"], keep="first")  # Devuelta tiene prioridad

print(f"\n✅ TOTAL IDs con TARGET: {len(df_ids):,}")
print("\nDistribución del TARGET:")
dist = df_ids["target"].value_counts().sort_index()
for clase, count in dist.items():
    label = {0: "Auditada", 1: "Glosada", 2: "Devuelta"}[clase]
    print(f"  Clase {clase} ({label}): {count:,} ({count/len(df_ids)*100:.1f}%)")

In [ ]:
# Celda 3 — Paso 2: Extraer features de cm_facturas
# Solo columnas disponibles en el momento de la radicación (sin data leakage)

print("=" * 60)
print("PASO 2: Extrayendo features de cm_facturas")
print("=" * 60)

factura_ids = tuple(df_ids["factura_id"].tolist())

# Se consulta en chunks de 50.000 IDs para evitar queries demasiado largas
CHUNK_SIZE = 50_000
chunks_ids = [factura_ids[i:i+CHUNK_SIZE] for i in range(0, len(factura_ids), CHUNK_SIZE)]

print(f"  Total IDs: {len(factura_ids):,} | Chunks: {len(chunks_ids)}")

df_features_list = []

for i, chunk in enumerate(chunks_ids):
    ids_str = ",".join(str(x) for x in chunk)
    q = f"""
    SELECT
        f.id                          AS factura_id,
        f.nit                         AS nit_prestador,
        f.mae_tipo_contrato_codigo    AS tipo_contrato_codigo,
        f.mae_tipo_contrato_valor     AS tipo_contrato,
        f.mae_regimen_codigo          AS regimen_codigo,
        f.mae_regimen_valor           AS regimen,
        f.valor_factura,
        f.valor_copago,
        f.valor_cuota_moderadora,
        f.fecha_prestacion,
        f.fecha_radicacion,
        DATEDIFF(
            DATE(f.fecha_radicacion),
            DATE(f.fecha_prestacion)
        )                             AS dias_hasta_radicacion,
        f.numero_contrato,
        f.origen_factura,
        f.pbs,
        f.cm_fe_rips_cargas_id,
        MONTH(f.fecha_radicacion)     AS mes_radicacion,
        QUARTER(f.fecha_radicacion)   AS trimestre_radicacion,
        YEAR(f.fecha_radicacion)      AS anio_radicacion
    FROM cm_facturas f
    WHERE f.id IN ({ids_str})
    """
    df_chunk = pd.read_sql(q, con=engine)
    df_features_list.append(df_chunk)
    print(f"  Chunk {i+1}/{len(chunks_ids)}: {len(df_chunk):,} filas")

df_features_facturas = pd.concat(df_features_list, ignore_index=True)
print(f"\n✅ Features cm_facturas: {len(df_features_facturas):,} filas × {len(df_features_facturas.columns)} cols")

In [ ]:
# Celda 4 — Paso 3: Extraer features agregadas de cm_detalles
# Agrega por factura: conteos de ítems, CUPS, diagnósticos y valores
#
# NOTA: codigo_dx tiene NULLs masivos en cm_detalles.
# Se extrae el código CIE-10 del prefijo de nombre_dx (e.g. "K083-RAIZ DENTAL RETENIDA" → "K083")
# usando SUBSTRING_INDEX(nombre_dx, '-', 1).

print("=" * 60)
print("PASO 3: Extrayendo features agregadas de cm_detalles")
print("=" * 60)

df_detalles_list = []

for i, chunk in enumerate(chunks_ids):
    ids_str = ",".join(str(x) for x in chunk)
    q = f"""
    SELECT
        cd.cm_facturas_id                                       AS factura_id,
        COUNT(*)                                                AS num_items,
        COUNT(DISTINCT cd.ma_servicio_codigo)                   AS num_cups_distintos,
        COUNT(DISTINCT
            NULLIF(SUBSTRING_INDEX(cd.nombre_dx, '-', 1), '')
        )                                                       AS num_dx_distintos,
        SUM(cd.valor_facturado)                                 AS sum_valor_facturado,
        MAX(cd.valor_facturado)                                 AS max_valor_item,
        AVG(cd.valor_facturado)                                 AS avg_valor_item,
        SUM(COALESCE(cd.valor_copago_item, 0))                  AS sum_copago_items,
        SUM(CASE WHEN cd.aplica_pbs = 1 THEN 1 ELSE 0 END)     AS num_items_pbs,
        SUM(CASE WHEN cd.aplica_autorizacion = 1
                 THEN 1 ELSE 0 END)                            AS num_items_con_autorizacion,
        COUNT(DISTINCT cd.tipo_servicio)                        AS num_tipos_servicio,
        COUNT(DISTINCT cd.mae_ambito_codigo)                    AS num_ambitos_distintos
    FROM cm_detalles cd
    WHERE cd.cm_facturas_id IN ({ids_str})
    GROUP BY cd.cm_facturas_id
    """
    df_chunk = pd.read_sql(q, con=engine)
    df_detalles_list.append(df_chunk)
    print(f"  Chunk {i+1}/{len(chunks_ids)}: {len(df_chunk):,} facturas")

df_features_detalles = pd.concat(df_detalles_list, ignore_index=True)
print(f"\n✅ Features cm_detalles: {len(df_features_detalles):,} filas × {len(df_features_detalles.columns)} cols")

# Verificar que num_dx_distintos ya no es todo ceros
dx_max = df_features_detalles["num_dx_distintos"].max()
dx_nonzero = (df_features_detalles["num_dx_distintos"] > 0).sum()
print(f"\n  num_dx_distintos → max={dx_max}  filas con dx>0: {dx_nonzero:,}")

In [ ]:
# Celda 5 — Construir features históricas del prestador
# Tasa histórica de glosas y devoluciones por NIT (últimos 24 meses)
# IMPORTANTE: Se calcula SOLO sobre el período ANTERIOR a la fecha de radicación
# para evitar data leakage. Aquí usamos una aproximación global para la primera versión.

print("=" * 60)
print("PASO 4: Calculando features históricas por NIT prestador")
print("=" * 60)

nits_str = "'" + "','".join(df_features_facturas["nit_prestador"].dropna().unique().tolist()) + "'"

q_provider_stats = f"""
SELECT
    f.nit                                           AS nit_prestador,
    COUNT(DISTINCT f.id)                            AS total_facturas_nit,
    COUNT(DISTINCT cad.cm_facturas_id)              AS total_devueltas_nit,
    ROUND(
        COUNT(DISTINCT cad.cm_facturas_id) * 100.0
        / NULLIF(COUNT(DISTINCT f.id), 0), 2
    )                                               AS tasa_devolucion_nit,
    ROUND(AVG(f.valor_factura), 0)                  AS valor_promedio_nit,
    ROUND(AVG(
        DATEDIFF(DATE(f.fecha_radicacion), DATE(f.fecha_prestacion))
    ), 1)                                           AS dias_radicacion_promedio_nit
FROM cm_facturas f
LEFT JOIN cm_auditoria_devoluciones cad ON cad.cm_facturas_id = f.id
WHERE f.fecha_radicacion >= DATE_SUB(NOW(), INTERVAL 24 MONTH)
  AND f.nit IN ({nits_str})
GROUP BY f.nit
"""

print("  Calculando estadísticas por NIT (puede tardar 1-2 min)...")
t0 = time.time()
df_provider_stats = pd.read_sql(q_provider_stats, con=engine)
print(f"  → {len(df_provider_stats):,} NITs procesados ({time.time()-t0:.1f}s)")

# Calcular tasa de glosas por NIT (requiere JOIN con cm_detalles)
q_tasa_glosas = f"""
SELECT
    f.nit                                           AS nit_prestador,
    COUNT(DISTINCT f.id)                            AS facturas_con_glosa_nit,
    ROUND(SUM(camg_agg.valor_glosa_total), 0)       AS valor_glosa_total_nit
FROM cm_facturas f
INNER JOIN (
    SELECT cd.cm_facturas_id,
           SUM(camg.valor_motivo) AS valor_glosa_total
    FROM cm_detalles cd
    INNER JOIN cm_auditoria_motivos_glosas camg ON camg.cm_detalles_id = cd.id
    GROUP BY cd.cm_facturas_id
) camg_agg ON camg_agg.cm_facturas_id = f.id
WHERE f.fecha_radicacion >= DATE_SUB(NOW(), INTERVAL 24 MONTH)
  AND f.nit IN ({nits_str})
GROUP BY f.nit
"""

print("  Calculando tasa de glosas por NIT (puede tardar 3-5 min)...")
t0 = time.time()
df_tasa_glosas = pd.read_sql(q_tasa_glosas, con=engine)
print(f"  → {len(df_tasa_glosas):,} NITs con glosas ({time.time()-t0:.1f}s)")

# Unir estadísticas del prestador
df_provider = df_provider_stats.merge(df_tasa_glosas, on="nit_prestador", how="left")
df_provider["tasa_glosa_nit"] = (
    df_provider["facturas_con_glosa_nit"].fillna(0) * 100
    / df_provider["total_facturas_nit"].replace(0, np.nan)
).round(2)

print(f"\n✅ Provider stats: {len(df_provider):,} NITs")

In [ ]:
# Celda 6 — Ensamblar el dataset final

print("=" * 60)
print("PASO 5: Ensamblando dataset final")
print("=" * 60)

# JOIN: IDs con target + features cm_facturas
df = df_ids.merge(df_features_facturas, on="factura_id", how="left")
print(f"  Después de JOIN con cm_facturas: {len(df):,} filas")

# JOIN: + features cm_detalles
df = df.merge(df_features_detalles, on="factura_id", how="left")
print(f"  Después de JOIN con cm_detalles: {len(df):,} filas")

# JOIN: + features del prestador
df = df.merge(df_provider[["nit_prestador", "total_facturas_nit", "tasa_devolucion_nit",
                            "tasa_glosa_nit", "valor_promedio_nit", "dias_radicacion_promedio_nit"]],
              on="nit_prestador", how="left")
print(f"  Después de JOIN con provider stats: {len(df):,} filas")

# Feature derivada: porcentaje del item de mayor valor sobre el total
df["porcentaje_mayor_item"] = (
    df["max_valor_item"] / df["valor_factura"].replace(0, np.nan)
).round(4)

# Feature derivada: discrepancia entre sum_detalles y valor_factura
df["discrepancia_valor"] = (
    df["sum_valor_facturado"] - df["valor_factura"]
).abs()

# Tiene FEV electrónica
df["tiene_fev"] = df["cm_fe_rips_cargas_id"].notna().astype(int)

# Asegurar que dias_hasta_radicacion no sea negativa (error de datos)
df["dias_hasta_radicacion"] = df["dias_hasta_radicacion"].clip(lower=0)

print(f"\n✅ Dataset ensamblado: {len(df):,} filas × {len(df.columns)} columnas")
print(f"\nDistribución final del TARGET:")
dist = df["target"].value_counts().sort_index()
for clase, count in dist.items():
    label = {0: "Auditada", 1: "Glosada", 2: "Devuelta"}[clase]
    print(f"  Clase {clase} ({label:8s}): {count:>8,} ({count/len(df)*100:.1f}%)")

In [ ]:
# Celda 7 — Calidad del dataset

print("=" * 60)
print("CALIDAD DEL DATASET")
print("=" * 60)

# Valores nulos
nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(1)
df_nulls = pd.DataFrame({"nulos": nulls, "pct": nulls_pct})
df_nulls = df_nulls[df_nulls["nulos"] > 0].sort_values("pct", ascending=False)

print(f"\nColumnas con nulos ({len(df_nulls)} de {len(df.columns)}):")
print(df_nulls.to_string())

# Tipos de datos
print(f"\nTipos de datos:")
print(df.dtypes.to_string())

# Estadísticas descriptivas columnas numéricas
print(f"\nEstadísticas descriptivas (columnas numéricas):")
display(df.select_dtypes(include=["number"]).describe().T)

In [ ]:
# Celda 8 — Limpieza y preparación final

print("=" * 60)
print("LIMPIEZA Y PREPARACIÓN FINAL")
print("=" * 60)

# Eliminar columnas que NO son features ni identifiadores
COLS_EXCLUIR = [
    "cm_fe_rips_cargas_id",  # Reemplazada por tiene_fev
    "numero_contrato",        # Alta cardinalidad, usar como feature categórica solo si es necesario
]
df_final = df.drop(columns=[c for c in COLS_EXCLUIR if c in df.columns])

# Codificar tipo de contrato (evitar strings en el modelo)
df_final["tipo_contrato_encoded"] = df_final["tipo_contrato"].map({
    "CAPITA": 0, "Evento": 1, "EVENTO": 1, "Paquete": 2, "PAQUETE": 2
}).fillna(-1).astype(int)

df_final["regimen_encoded"] = df_final["regimen"].map({
    "SUBSIDIADO": 0, "CONTRIBUTIVO": 1
}).fillna(-1).astype(int)

# Rellenar nulos numéricos con 0 (facturas sin detalles = inconsistencia → puede ser señal)
cols_numericas = df_final.select_dtypes(include=["number"]).columns.tolist()
df_final[cols_numericas] = df_final[cols_numericas].fillna(0)

print(f"Dataset final: {len(df_final):,} filas × {len(df_final.columns)} columnas")
print(f"\nColumnas finales:")
for col in df_final.columns:
    print(f"  {col}: {df_final[col].dtype}")

In [ ]:
# Celda 9 — Guardar como Parquet

print("=" * 60)
print("GUARDANDO DATASET")
print("=" * 60)

df_final.to_parquet(OUTPUT_FILE, index=False, engine="pyarrow", compression="snappy")

size_mb = OUTPUT_FILE.stat().st_size / 1024 / 1024
print(f"✅ Guardado: {OUTPUT_FILE}")
print(f"   Tamaño: {size_mb:.1f} MB")
print(f"   Filas:  {len(df_final):,}")
print(f"   Cols:   {len(df_final.columns)}")

# Verificar que se puede leer
df_check = pd.read_parquet(OUTPUT_FILE)
print(f"\n✅ Verificación lectura OK: {len(df_check):,} filas")

print("\n" + "=" * 60)
print("DISTRIBUCIÓN FINAL DE CLASES")
print("=" * 60)
dist = df_final["target"].value_counts().sort_index()
for clase, count in dist.items():
    label = {0: "Auditada", 1: "Glosada", 2: "Devuelta"}[clase]
    barra = "█" * int(count / max(dist.values) * 30)
    print(f"  {clase} ({label:8s}): {barra} {count:>8,} ({count/len(df_final)*100:.1f}%)")

print("\n" + "=" * 60)
print("PRÓXIMOS PASOS")
print("=" * 60)
print("  1. notebooks/03_EDA.ipynb — Análisis exploratorio del dataset")
print("  2. Verificar distribución de valores en columnas críticas")
print("  3. Aplicar Feature Engineering (src/features/)")
print("  4. Entrenamiento del modelo (Fase 3)")

In [1]:
# Celda 10 — CORRECCIÓN DEL PARQUET EXISTENTE (versión optimizada)
# Parchea el dataset ya guardado sin re-correr toda la extracción.
#
# Problemas corregidos:
#   1. num_dx_distintos = 0 → código CIE-10 estaba en nombre_dx, no en codigo_dx
#   2. sum_copago_items = NULL → aplica COALESCE(valor_copago_item, 0)
#   3. dias_hasta_radicacion outliers extremos (max 4045 días) → clip a 365
#   4. porcentaje_mayor_item > 1 → cap a 1.0
#   5. valor_factura outliers extremos (> p99.9) → winsorize
#
# OPTIMIZACIÓN: En lugar de 15 chunks con WHERE IN (50K IDs) sobre 260M filas,
# se hace UNA sola query filtrando por rango de fecha (usa índice de cm_facturas)
# y luego se filtra en Python. Esto es 10-20x más rápido.

import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from pathlib import Path
import time

print("=" * 60)
print("CORRECCIÓN DEL PARQUET EXISTENTE")
print("=" * 60)

OUTPUT_FILE = Path(r"D:\Users\jcardonr\Documents\eafit\Codigo\data\processed\20260304_dataset_entrenamiento.parquet")

user, password, host, db_name, port = "jcardonr", "6nxx2yVWaKXyiF1cZbfS*", "10.250.5.36", "system_savia", "3306"
engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{db_name}",
    connect_args={"connect_timeout": 600},
    pool_recycle=3600,
)

# ── 1. Cargar Parquet existente ────────────────────────────────
df = pd.read_parquet(OUTPUT_FILE)
ids_set = set(df["factura_id"].tolist())
print(f"[CARGA] {len(df):,} filas × {len(df.columns)} cols")

# ── 2. UNA sola query optimizada: JOIN por fecha (no IN) ──────
# Filtra en cm_facturas por fecha_radicacion (índice disponible).
# Devuelve num_dx y sum_copago para TODAS las facturas en el período.
# Luego filtramos en Python a nuestros 728K IDs.
print("\n[FIX 1 & 2] Re-extrayendo dx y copago via JOIN por fecha (1 sola query)...")
print("  Estimado: 5-15 min (vs 146 min anterior con chunks de IN)")

q_corr = """
SELECT
    cd.cm_facturas_id                                       AS factura_id,
    COUNT(DISTINCT
        NULLIF(SUBSTRING_INDEX(cd.nombre_dx, '-', 1), '')
    )                                                       AS num_dx_distintos,
    SUM(COALESCE(cd.valor_copago_item, 0))                  AS sum_copago_items
FROM cm_detalles cd
INNER JOIN cm_facturas f ON f.id = cd.cm_facturas_id
WHERE f.fecha_radicacion >= DATE_SUB(NOW(), INTERVAL 24 MONTH)
GROUP BY cd.cm_facturas_id
"""

t0 = time.time()
df_corr = pd.read_sql(q_corr, con=engine)
elapsed = time.time() - t0
print(f"  ✅ Query completada en {elapsed/60:.1f} min  →  {len(df_corr):,} facturas")

# Filtrar solo nuestros IDs
df_corr = df_corr[df_corr["factura_id"].isin(ids_set)].reset_index(drop=True)
print(f"  Después de filtrar a nuestros IDs: {len(df_corr):,}")
print(f"  num_dx_distintos → max={df_corr['num_dx_distintos'].max()}, "
      f"filas>0: {(df_corr['num_dx_distintos']>0).sum():,}")
print(f"  sum_copago_items → max={df_corr['sum_copago_items'].max():,.0f}, "
      f"filas>0: {(df_corr['sum_copago_items']>0).sum():,}")

# Aplicar al dataframe principal
df = df.drop(columns=["num_dx_distintos", "sum_copago_items"], errors="ignore")
df = df.merge(df_corr, on="factura_id", how="left")
df[["num_dx_distintos", "sum_copago_items"]] = df[["num_dx_distintos", "sum_copago_items"]].fillna(0)

# ── 3. Clip dias_hasta_radicacion ─────────────────────────────
antes = df["dias_hasta_radicacion"].max()
df["dias_hasta_radicacion"] = df["dias_hasta_radicacion"].clip(lower=0, upper=365)
n_clipped = (df["dias_hasta_radicacion"] == 365).sum()
print(f"\n[FIX 3] dias_hasta_radicacion: max {antes:.0f} → cap 365  ({n_clipped:,} filas afectadas)")

# ── 4. Recalcular porcentaje_mayor_item con cap 1.0 ──────────
df["porcentaje_mayor_item"] = (
    df["max_valor_item"] / df["valor_factura"].replace(0, np.nan)
).clip(upper=1.0).round(4).fillna(0)
print(f"[FIX 4] porcentaje_mayor_item → max={df['porcentaje_mayor_item'].max():.4f}")

# ── 5. Winsorize valor_factura al p99.9 ───────────────────────
p999 = df["valor_factura"].quantile(0.999)
n_outliers = (df["valor_factura"] > p999).sum()
df["valor_factura_original"] = df["valor_factura"]
df["valor_factura"] = df["valor_factura"].clip(upper=p999)
print(f"[FIX 5] valor_factura: p99.9=${p999:,.0f}  winsorizadas={n_outliers:,}")

# ── 6. Guardar ────────────────────────────────────────────────
df.to_parquet(OUTPUT_FILE, index=False, engine="pyarrow", compression="snappy")
size_mb = OUTPUT_FILE.stat().st_size / 1024 / 1024

print(f"\n✅ Parquet corregido guardado")
print(f"   Tamaño: {size_mb:.1f} MB  |  Filas: {len(df):,}  |  Cols: {len(df.columns)}")

print("\nEstadísticas post-corrección:")
for c in ["num_dx_distintos", "sum_copago_items", "dias_hasta_radicacion",
          "porcentaje_mayor_item", "valor_factura"]:
    s = df[c]
    print(f"  {c}: min={s.min():.1f}  max={s.max():.1f}  media={s.mean():.2f}")

CORRECCIÓN DEL PARQUET EXISTENTE
[CARGA] 728,289 filas × 39 cols

[FIX 1 & 2] Re-extrayendo dx y copago via JOIN por fecha (1 sola query)...
  Estimado: 5-15 min (vs 146 min anterior con chunks de IN)
  ✅ Query completada en 17.8 min  →  4,500,583 facturas
  Después de filtrar a nuestros IDs: 725,983
  num_dx_distintos → max=2531, filas>0: 656,506
  sum_copago_items → max=0, filas>0: 0

[FIX 3] dias_hasta_radicacion: max 4045 → cap 365  (19,024 filas afectadas)
[FIX 4] porcentaje_mayor_item → max=1.0000
[FIX 5] valor_factura: p99.9=$348,497,204  winsorizadas=729

✅ Parquet corregido guardado
   Tamaño: 26.5 MB  |  Filas: 728,289  |  Cols: 40

Estadísticas post-corrección:
  num_dx_distintos: min=0.0  max=2531.0  media=2.36
  sum_copago_items: min=0.0  max=0.0  media=0.00
  dias_hasta_radicacion: min=0.0  max=365.0  media=43.49
  porcentaje_mayor_item: min=0.0  max=1.0  media=0.73
  valor_factura: min=0.0  max=348497203.7  media=3447007.26


In [2]:
# Celda 11 — Limpieza final: eliminar sum_copago_items (siempre 0 en BD)
# valor_copago_item en cm_detalles está vacío; el copago se registra en cm_facturas.valor_copago
# que ya está en el dataset. sum_copago_items es redundante → se elimina.

import pandas as pd
from pathlib import Path

OUTPUT_FILE = Path(r"D:\Users\jcardonr\Documents\eafit\Codigo\data\processed\20260304_dataset_entrenamiento.parquet")

df = pd.read_parquet(OUTPUT_FILE)
print(f"Antes: {len(df.columns)} columnas")

df = df.drop(columns=["sum_copago_items"], errors="ignore")
print(f"Después: {len(df.columns)} columnas  (eliminada sum_copago_items)")

# Verificar columnas finales del dataset
print("\nColumnas del dataset final:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}: {df[col].dtype}")

df.to_parquet(OUTPUT_FILE, index=False, engine="pyarrow", compression="snappy")
print(f"\n✅ Parquet actualizado: {OUTPUT_FILE.stat().st_size/1024/1024:.1f} MB  |  {len(df):,} filas × {len(df.columns)} cols")
print("\n🎯 Dataset listo para EDA → notebooks/03_EDA.ipynb")

Antes: 40 columnas
Después: 39 columnas  (eliminada sum_copago_items)

Columnas del dataset final:
   1. factura_id: int64
   2. target: int64
   3. nit_prestador: object
   4. tipo_contrato_codigo: object
   5. tipo_contrato: object
   6. regimen_codigo: object
   7. regimen: object
   8. valor_factura: float64
   9. valor_copago: float64
  10. valor_cuota_moderadora: float64
  11. fecha_prestacion: datetime64[ns]
  12. fecha_radicacion: datetime64[ns]
  13. dias_hasta_radicacion: int64
  14. origen_factura: int64
  15. pbs: object
  16. mes_radicacion: int64
  17. trimestre_radicacion: int64
  18. anio_radicacion: int64
  19. num_items: float64
  20. num_cups_distintos: float64
  21. sum_valor_facturado: float64
  22. max_valor_item: float64
  23. avg_valor_item: float64
  24. num_items_pbs: float64
  25. num_items_con_autorizacion: float64
  26. num_tipos_servicio: float64
  27. num_ambitos_distintos: float64
  28. total_facturas_nit: int64
  29. tasa_devolucion_nit: float64
  30. t